# Data processing

In this notebook, we will work with the data set created from the initial spatial merge and perform various checks to ensure its integrity. This will also involve some data processing. First, we must load the data set.

## 1. Understanding the Data Set

In [2]:
# Imports
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from paths import *

In [3]:
df = pd.read_csv(INT_DATA_PATH / "sopp_svi_merged.csv")
print(df.shape)
df.head()

(383027, 24)


,Unnamed: 0,raw_row_number,date,time,service_area,subject_age,subject_race,subject_sex,type,arrest_made,...,search_conducted,search_person,search_vehicle,search_basis,reason_for_search,reason_for_stop,raw_action_taken,raw_subject_race_description,serv,svi_rpl_themes
0,0,1,2014-01-01,01:25:00,110,24.0,white,male,vehicular,False,...,False,False,False,NaN,NaN,Moving Violation,Citation,WHITE,110.0,0.241848
1,1,2,2014-01-01,05:47:00,320,42.0,white,male,vehicular,False,...,False,False,False,NaN,NaN,Moving Violation,Verbal Warning,WHITE,320.0,0.213643
2,2,3,2014-01-01,07:46:00,320,29.0,asian/pacific islander,male,vehicular,False,...,False,False,False,NaN,NaN,Moving Violation,Verbal Warning,LAOTIAN,320.0,0.213643
3,3,4,2014-01-01,08:10:00,610,23.0,white,male,vehicular,False,...,False,False,False,NaN,NaN,Moving Violation,Citation,WHITE,610.0,0.121181
4,4,5,2014-01-01,08:35:00,930,35.0,hispanic,male,vehicular,False,...,False,False,False,NaN,NaN,Equipment Violation,Citation,HISPANIC,930.0,0.075382


## 2. Integrity

Basic checks

In [4]:
print("Rows, cols:", df.shape)
print("Exact duplicate rows:", df.duplicated().sum())

# column missingness
missing = df.isna().mean().sort_values(ascending=False)
print("Top missing columns:")
print((missing.head(5) * 100).round(2))

Rows, cols: (383027, 24)
Exact duplicate rows: 0
Top missing columns:
reason_for_search    96.27
search_basis         95.75
contraband_found     95.75
outcome              10.23
arrest_made           9.07
dtype: float64


## 3. Some Pre-processing
Normalizing string labels

In [5]:
def normalize_label(x):
    if pd.isna(x): 
        return x
    x = str(x).strip()
    x = re.sub(r"^[^\w]+", "", x)   # remove leading non-word chars (like "&")
    x = re.sub(r"\s+", " ", x)      # collapse repeated whitespace
    return x

obj_cols = df.select_dtypes(include=["object", "str"]).columns
for c in obj_cols:
    df[c] = df[c].map(normalize_label)

Here, we parse some temporal features to use during exploratory data analysis. These are not guaranteed to be used during modeling. We also create a binned version of the hour of day column to aid in deploying the logistic model.

In [6]:
# date -> day_of_week
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["day_of_week"] = df["date"].dt.day_name()

# time -> hour (handles HH:MM:SS and HH:MM)
t1 = pd.to_datetime(df["time"], format="%H:%M:%S", errors="coerce")
t2 = pd.to_datetime(df["time"], format="%H:%M", errors="coerce")
df["hour"] = t1.fillna(t2).dt.hour

# hour bins
bins = [-0.5, 6.5, 9.5, 14.5, 17.5, 23.5]
labels = ["12AM-6AM", "7AM-9AM", "10AM-2PM", "3PM-5PM", "6PM-11PM"]
df["hour_bin"] = pd.cut(df["hour"], bins=bins, labels=labels)
df["hour_bin"] = pd.Categorical(df["hour_bin"], categories=labels, ordered=True)

There are many variables in this data set that would be invalid as a predictor of searches. We group these into two categories: irrelevant variables and those which inform post-stop decisions.

The reason for the first's invalidity is that these variables carry no interpretation and were generally created due to intermediate processing steps. The latter is invalid because using information of events that occured after the officer has already decided whether to conduct a search is a form of data leakage and would bias any models we fit.

In [7]:
# Drop post-stop information since it would cause data leakage
drop_post_stop = [
    "contraband_found",     # Recorded after search
    "search_basis",         # Recorded after search
    "reason_for_search",    # Recorded after search
    "search_person",        # Recorded after search
    "search_vehicle",       # Recorded after search
    "outcome",              # Downstream outcome
    "arrest_made",          # Downstream outcome
    "citation_issued",      # Downstream outcome
    "warning_issued",       # Downstream outcome
    "raw_action_taken"      # Downstream outcome
]

drop_irrelevant = [
    "Unnamed: 0",                   # Row id from data wrangling
    "raw_row_number",               # Row id from data wrangling
    "raw_subject_race_description", # Redundant with subject_race
    "date",                         # Redundant with engineered columns
    "time",                         # Redundant with engineered columns
    "serv",                         # Redundant with service_area
    "type",                         # No variance in column. Poor predictor.
]

In [8]:
drop_cols = list(set(drop_post_stop + drop_irrelevant))
print("\nDropping {} columns total:".format(len(drop_cols)))
df = df.drop(columns=drop_cols)

print("\nReduced df shape:", df.shape)
print("Remaining columns:")
print(df.columns)
print("\nPreview of the remaining data:")
print(df.head())


Dropping 17 columns total:

Reduced df shape: (383027, 10)
Remaining columns:
Index(['service_area', 'subject_age', 'subject_race', 'subject_sex',
       'search_conducted', 'reason_for_stop', 'svi_rpl_themes', 'day_of_week',
       'hour', 'hour_bin'],
      dtype='str')

Preview of the remaining data:
  service_area  subject_age            subject_race subject_sex  \
0          110         24.0                   white        male   
1          320         42.0                   white        male   
2          320         29.0  asian/pacific islander        male   
3          610         23.0                   white        male   
4          930         35.0                hispanic        male   

   search_conducted      reason_for_stop  svi_rpl_themes day_of_week  hour  \
0             False     Moving Violation        0.241848   Wednesday   1.0   
1             False     Moving Violation        0.213643   Wednesday   5.0   
2             False     Moving Violation        0.213643 

Now, we encode categorical columns

In [9]:
# Binary outcome
df["search_conducted"] = df["search_conducted"].astype(int)
df["subject_sex"] = df["subject_sex"].map({"female": 0, "male": 1})

# >= 3 levels
cat_cols = [
    "subject_race",
    "year",
    "month",
    "reason_for_stop",
    "service_area",
    "day_of_week",
]

cat_cols = [c for c in cat_cols if c in df.columns]

# Normalize labels for strings and convert to category
for c in cat_cols:
    df[c] = df[c].astype("category")

print("Converted to category:", cat_cols)

# Inspect levels + counts
for c in cat_cols:
    print(f"\n=== {c} ===")
    print("dtype:", df[c].dtype)
    print("n_unique:", df[c].nunique(dropna=True))
    print("missing %:", round(df[c].isna().mean() * 100, 3))

Converted to category: ['subject_race', 'reason_for_stop', 'service_area', 'day_of_week']

=== subject_race ===
dtype: category
n_unique: 5
missing %: 0.322

=== reason_for_stop ===
dtype: category
n_unique: 94
missing %: 0.057

=== service_area ===
dtype: category
n_unique: 25
missing %: 0.0

=== day_of_week ===
dtype: category
n_unique: 7
missing %: 0.048


As we can see, there are over 90 observed reasons for stop. Creating 96 dummy variables may lead to a less powerful model fit. It seems that most of the occurances are one of 6 stops. So, we will classify all others as being in the category `Other`.

In [10]:
K = 6
r = df["reason_for_stop"].astype("object")
topk = r.value_counts(dropna=True).head(K).index.tolist()
print("Top K reasons for stop:", topk)

# Classify everything else as "Other"
df["reason_for_stop"] = np.where(
    r.isna(),
    np.nan,
    np.where(r.isin(topk), r, "Other")
)

# Convert back to category
df["reason_for_stop"] = df["reason_for_stop"].astype("category")
df["reason_for_stop"] = df["reason_for_stop"].cat.remove_unused_categories()

print("\nNew reason_for_stop counts:")
print(df["reason_for_stop"].value_counts(dropna=False))
print("\nNumber of levels:", df["reason_for_stop"].nunique(dropna=True))

Top K reasons for stop: ['Moving Violation', 'Equipment Violation', 'Radio Call/Citizen Contact', 'Muni, County, H&S Code', 'Personal Knowledge/Informant', 'Suspect Info (I.S., Bulletin, Log)']

New reason_for_stop counts:
reason_for_stop
Moving Violation                      279856
Equipment Violation                    97375
Radio Call/Citizen Contact              1887
Muni, County, H&S Code                  1291
Other                                   1013
Personal Knowledge/Informant             860
Suspect Info (I.S., Bulletin, Log)       526
NaN                                      219
Name: count, dtype: int64

Number of levels: 7


Next, we will handle missingness in the remaining data set. Above, we can see that many of the columns are full. If the amount of data present by removing records that contain *any* missing values is sufficient, we may proceed by only retaining complete cases. This appears to be the case.

In [11]:
print("Before complete-case:", df.shape)
df_cc = df.dropna()
print("After complete-case:", df_cc.shape)

Before complete-case: (383027, 10)
After complete-case: (358690, 10)


Here, we are making some final quality of life changes. We will rename the column containing the SVI information and change the data type for the subject sex column.

In [12]:
df_cc = df_cc.rename(columns={"svi_rpl_themes": "svi"})
df_cc["subject_sex"] = pd.Categorical(df_cc["subject_sex"]).codes.astype("int64")

Finally, we export the final data set.

In [13]:
df_cc.to_csv(FINAL_DATA_PATH / "post_eda_df.csv")